## Importación de los datos

In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from scipy import stats
import os
import sys
!{sys.executable} -m pip install pyproj folium
from pyproj import Transformer
import folium

In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

BASE_URL = "https://ckan.montevideo.gub.uy"
DATASET_ID_PM25 = "red-de-monitoreo-de-la-calidad-del-aire-de-montevideo"

r_PM25 = requests.get(f"{BASE_URL}/api/3/action/package_show", params={"id": DATASET_ID_PM25}, headers=headers)
resources_PM25 = r_PM25.json()["result"]["resources"]

dfs_pm25 = []

for res in resources_PM25:
    es_csv = "CSV" in res["format"].upper() # Anteriormente usé "res["format"].upper() == "CSV"" pero esto excluia los CSV del 2014, 2022, 2023 porque tienen terminación .CSV 
    es_metadato = "aire-" in res["url"]
    es_historico_manual = "manuales" in res["url"]

    if es_csv and not es_metadato and not es_historico_manual:
        try:
            df_temp = pd.read_csv(res["url"], storage_options=headers)
            if "pollutant_id" in df_temp.columns:
                df_pm2 = df_temp[df_temp["pollutant_id"] == "PM2"].copy()
                df_pm2["archivo_origen"] = res["name"]
                dfs_pm25.append(df_pm2)
                print(f"{res['name']}: {len(df_pm2)} filas de PM2 (de {len(df_temp)} totales)")
        except Exception as e:
            print(f"Error con {res['name']}: {e}")

pm25_completo = pd.concat(dfs_pm25, ignore_index=True)
print("\nTotal filas de PM2:", len(pm25_completo)) # Esto es más una formalidad que otra cosa
pm25_completo.head()

Medidas de la calidad del aire - 2014: 0 filas de PM2 (de 13422 totales)
Medidas de la calidad del aire - 2015: 8688 filas de PM2 (de 25838 totales)
Medidas de la calidad del aire - 2016: 8784 filas de PM2 (de 18377 totales)
Medidas de la calidad del aire - 2017: 18448 filas de PM2 (de 49383 totales)
Medidas de la calidad del aire - 2018: 26244 filas de PM2 (de 71121 totales)
Medidas de la calidad del aire - 2019: 23028 filas de PM2 (de 69936 totales)
Medidas de la calidad del aire - 2020: 26520 filas de PM2 (de 79056 totales)
Medidas de la calidad del aire - 2021: 26280 filas de PM2 (de 70080 totales)
Medidas de la calidad del aire - 2022: 26280 filas de PM2 (de 70080 totales)
Medidas de la calidad del aire - 2023: 26280 filas de PM2 (de 70080 totales)
Medidas de la calidad del aire - 2024: 26352 filas de PM2 (de 70272 totales)
Medidas de la calidad del aire – 2025: 26280 filas de PM2 (de 61320 totales)

Total filas de PM2: 243184


,pollutant_id,pollutant_averaging,date,pollutant_value,pollutant_unit,station_id,X,Y,ID_estacion,method_id,archivo_origen,Unnamed: 0
0,PM2,1,2015-01-04 00:00:00,8.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN
1,PM2,1,2015-01-04 01:00:00,7.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN
2,PM2,1,2015-01-04 02:00:00,6.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN
3,PM2,1,2015-01-04 03:00:00,4.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN
4,PM2,1,2015-01-04 04:00:00,4.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN


## Exploratory Data Analysis

In [3]:
print(pm25_completo['archivo_origen'].value_counts())

archivo_origen
Medidas de la calidad del aire - 2020    26520
Medidas de la calidad del aire - 2024    26352
Medidas de la calidad del aire - 2021    26280
Medidas de la calidad del aire - 2022    26280
Medidas de la calidad del aire - 2023    26280
Medidas de la calidad del aire – 2025    26280
Medidas de la calidad del aire - 2018    26244
Medidas de la calidad del aire - 2019    23028
Medidas de la calidad del aire - 2017    18448
Medidas de la calidad del aire - 2016     8784
Medidas de la calidad del aire - 2015     8688
Name: count, dtype: int64


In [4]:
pm25_completo.info()
print("-----------------------------------")
print(f"El shape es: {pm25_completo.shape}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 243184 entries, 0 to 243183
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   pollutant_id         243184 non-null  object 
 1   pollutant_averaging  243184 non-null  int64  
 2   date                 243184 non-null  object 
 3   pollutant_value      213961 non-null  float64
 4   pollutant_unit       243184 non-null  object 
 5   station_id           243184 non-null  object 
 6   X                    243184 non-null  int64  
 7   Y                    243184 non-null  int64  
 8   ID_estacion          243184 non-null  object 
 9   method_id            243184 non-null  object 
 10  archivo_origen       243184 non-null  object 
 11  Unnamed: 0           75828 non-null   float64
dtypes: float64(2), int64(3), object(7)
memory usage: 22.3+ MB
-----------------------------------
El shape es: (243184, 12)


In [5]:
pm25_completo.head(1)

,pollutant_id,pollutant_averaging,date,pollutant_value,pollutant_unit,station_id,X,Y,ID_estacion,method_id,archivo_origen,Unnamed: 0
0,PM2,1,2015-01-04 00:00:00,8.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN


In [6]:
pm25_completo.tail(1)

,pollutant_id,pollutant_averaging,date,pollutant_value,pollutant_unit,station_id,X,Y,ID_estacion,method_id,archivo_origen,Unnamed: 0
243183,PM2,1,2025-12-31 23:00:00,27.0,ug/m3,UYMVD_E1,572452,6137044,Ciudad Vieja2,UYMVD_PM2_b,Medidas de la calidad del aire – 2025,NaN


In [7]:
pm25_completo.isna().sum()

pollutant_id                0
pollutant_averaging         0
date                        0
pollutant_value         29223
pollutant_unit              0
station_id                  0
X                           0
Y                           0
ID_estacion                 0
method_id                   0
archivo_origen              0
Unnamed: 0             167356
dtype: int64

In [8]:
# % de nulos por estación
print(pm25_completo.groupby("ID_estacion")['pollutant_value'].apply(lambda x: x.isna().mean() * 100))

ID_estacion
Ciudad Vieja 3       2.452999
Ciudad Vieja2        4.963284
Ciudad Vieja3        9.376620
Colon                3.264537
Curva de Maronas    14.753641
Tres Cruces 3       49.224884
Tres Cruces 4       14.746171
Name: pollutant_value, dtype: float64


In [9]:
# Cantidad de estaciones (ID-estacion)
print(f"Estaciones separadas por: {pm25_completo["ID_estacion"].value_counts()}")

Estaciones separadas por: ID_estacion
Curva de Maronas    78889
Tres Cruces 4       48182
Ciudad Vieja 3      34733
Ciudad Vieja3       30864
Ciudad Vieja2       30504
Colon               10078
Tres Cruces 3        9934
Name: count, dtype: int64


In [10]:
# Cantidad de estaciones con identificadores únicos (station_id)
print(f"Estaciones separadas por: {pm25_completo["station_id"].value_counts()}")

Estaciones separadas por: station_id
UYMVD_E1    96101
UYMVD_E6    78889
UYMVD_E5    58116
UYMVD_E8    10078
Name: count, dtype: int64


In [11]:
coordenadas_de_estaciones = pm25_completo[["ID_estacion", "X", "Y"]].drop_duplicates()
coordenadas_de_estaciones

,ID_estacion,X,Y
0,Ciudad Vieja 3,572796,6137122
17472,Curva de Maronas,579229,6142255
26065,Colon,570970,6149046
27419,Ciudad Vieja 3,572796,6137123
35920,Curva de Maronas,579230,6142255
70924,Tres Cruces 3,576247,6138473
76432,Ciudad Vieja3,572796,6137123
98570,Tres Cruces 4,576324,6138361
160072,Ciudad Vieja2,572452,6137044


In [14]:
# Según los metadatos de la intendencia: "aire-estaciones.csv", 
# las estaciones "Ciudad Vieja3" y "Ciudad Vieja2"  tienen un espacio entre el número y el nombre.
# Por ello, conviene normalizar los nombres para que coincidan con los metadatos

pm25_completo.loc[pm25_completo["ID_estacion"] == "Ciudad Vieja3", "ID_estacion"] = "Ciudad Vieja 3" 
pm25_completo.loc[pm25_completo["ID_estacion"] == "Ciudad Vieja2", "ID_estacion"] = "Ciudad Vieja 2"
pm25_completo["ID_estacion"].value_counts()

ID_estacion
Curva de Maronas    78889
Ciudad Vieja 3      65597
Tres Cruces 4       48182
Ciudad Vieja 2      30504
Colon               10078
Tres Cruces 3        9934
Name: count, dtype: int64

In [15]:
# Tengo dudas de porqué hay varias estaciones de "Ciudad vieja" o "Tres Cruces" con algunas centenas de metros de diferencia
# Por esto mismo, se intenta graficar usando 'folium' para tener una referencia visual

transformador = Transformer.from_crs("EPSG:32721", "EPSG:4326", always_xy=True)

coordenadas_de_estaciones["longitud"], coordenadas_de_estaciones["latitud"] = transformador.transform(
    coordenadas_de_estaciones["X"].values,
    coordenadas_de_estaciones["Y"].values
)

coordenadas_de_estaciones

,ID_estacion,X,Y,longitud,latitud
0,Ciudad Vieja 3,572796,6137122,-56.203172,-34.905724
17472,Curva de Maronas,579229,6142255,-56.133249,-34.858960
26065,Colon,570970,6149046,-56.224168,-34.798337
27419,Ciudad Vieja 3,572796,6137123,-56.203172,-34.905715
35920,Curva de Maronas,579230,6142255,-56.133238,-34.858960
70924,Tres Cruces 3,576247,6138473,-56.165524,-34.893289
76432,Ciudad Vieja3,572796,6137123,-56.203172,-34.905715
98570,Tres Cruces 4,576324,6138361,-56.164671,-34.894293
160072,Ciudad Vieja2,572452,6137044,-56.206930,-34.906452


In [16]:
# El mapa no puede verse desde GitHub, pero veré de dejar un .jpg de él en el repo

mapa_estaciones = folium.Map(location=[-34.87, -56.17], zoom_start=12)

for _, fila in coordenadas_de_estaciones.drop_duplicates(subset="ID_estacion").iterrows():
    folium.Marker(
        location=[fila["latitud"], fila["longitud"]],
        popup=fila["ID_estacion"],
        tooltip=fila["ID_estacion"]
    ).add_to(mapa_estaciones)

mapa_estaciones

**Se asume que las estaciones Ciudad Vieja 3 y Ciedad Vieja 2 son diferentes, al igual que Tres Cruces 3 y Tres Cruces 4**

In [17]:
print(pm25_completo.groupby("ID_estacion")['pollutant_value'].apply(lambda x: x.isna().mean() * 100))

ID_estacion
Ciudad Vieja 2       4.963284
Ciudad Vieja 3       5.710627
Colon                3.264537
Curva de Maronas    14.753641
Tres Cruces 3       49.224884
Tres Cruces 4       14.746171
Name: pollutant_value, dtype: float64


In [18]:
# Respetando que el df está en inglés, procedo a crear una nueva columna llamada "Year" para averiguar sobre los nulos de Tres Cruces 3
pm25_completo["date"] = pd.to_datetime(pm25_completo["date"])
pm25_completo["Year"] = pm25_completo["date"].dt.year

#Procedemos a agrupar los nulos de "Tres Cruces 3" y observar el resultado
nulos_tc3 = pm25_completo[pm25_completo["ID_estacion"] == "Tres Cruces 3"].groupby(["Year"])["pollutant_value"].apply(lambda x: x.isna().mean() * 100)

#Procedemos a revisar cuántos años de observaciones tiene la estación "Tres Cruces 3"
filas_tc3 = pm25_completo[pm25_completo["ID_estacion"] == "Tres Cruces 3"].groupby(["Year"]).size()


print(f"La cantidad de nulos por año de observación es de: \n{nulos_tc3}\n ---------")
print(f"La cantidad de observaciones por año es: \n{filas_tc3}\n ---------")

# Con los resultados se decide que, con la cantidad de datos que puede aportar la estación, es mejor eliminarla
# Se podría juntar la estación "Tres Cruces 3" con "Tres Cruces 4" pero es poco honesto metodológicamente
# Todos los datos del 2020 son nulos, y los datos del 2019 son apenas 5508. 
# Por lo tanto, se decide eliminar los datos de esta estación ya que geográficamente es diferente que "Tres Cruces 4" por poco más de 100 metros
# Y además la cantidad de datos que aporta no es significativa

pm25 = pm25_completo[pm25_completo["ID_estacion"] != "Tres Cruces 3"]
print("Filas en pm25_completo:", len(pm25_completo))
print("Filas en pm25:", len(pm25))
print(f"Diferencia: {len(pm25_completo) - len(pm25)}\n ---------")
print(f"La cantidad de datos nulos son: \n{pm25.groupby("ID_estacion")['pollutant_value'].apply(lambda x: x.isna().mean() * 100)}")

# En caso de necesitarlo, pueden utilizar este dataset en donde es únicamente de la estación "Tres Cruces 3"
tres_cruces_3_excluido = pm25_completo[pm25_completo["ID_estacion"] == "Tres Cruces 3"]

La cantidad de nulos por año de observación es de: 
Year
2019      8.42411
2020    100.00000
Name: pollutant_value, dtype: float64
 ---------
La cantidad de observaciones por año es: 
Year
2019    5508
2020    4426
dtype: int64
 ---------
Filas en pm25_completo: 243184
Filas en pm25: 233250
Diferencia: 9934
 ---------
La cantidad de datos nulos son: 
ID_estacion
Ciudad Vieja 2       4.963284
Ciudad Vieja 3       5.710627
Colon                3.264537
Curva de Maronas    14.753641
Tres Cruces 4       14.746171
Name: pollutant_value, dtype: float64


In [19]:
# Conviene revisar los nulos restantes para saber si imputamos o eliminamos
# Es una cantidad de nulos pequeña en comparación a O3 pero conviene revisar de igual manera

pm25[pm25["pollutant_value"].isna()].groupby("ID_estacion").head(3)

,pollutant_id,pollutant_averaging,date,pollutant_value,pollutant_unit,station_id,X,Y,ID_estacion,method_id,archivo_origen,Unnamed: 0,Year
897,PM2,1,2015-02-10 09:00:00,NaN,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN,2015
898,PM2,1,2015-02-10 10:00:00,NaN,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN,2015
899,PM2,1,2015-02-10 11:00:00,NaN,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN,2015
18723,PM2,1,2017-02-22 12:00:00,NaN,ug/m3,UYMVD_E6,579229,6142255,Curva de Maronas,UYMVD_PM2_b,Medidas de la calidad del aire - 2017,NaN,2017
27119,PM2,1,2017-12-19 10:00:00,NaN,ug/m3,UYMVD_E8,570970,6149046,Colon,UYMVD_PM2_b,Medidas de la calidad del aire - 2017,NaN,2017
27120,PM2,1,2017-12-19 11:00:00,NaN,ug/m3,UYMVD_E8,570970,6149046,Colon,UYMVD_PM2_b,Medidas de la calidad del aire - 2017,NaN,2017
27121,PM2,1,2017-12-19 12:00:00,NaN,ug/m3,UYMVD_E8,570970,6149046,Colon,UYMVD_PM2_b,Medidas de la calidad del aire - 2017,NaN,2017
36052,PM2,1,2018-01-06 12:00:00,NaN,ug/m3,UYMVD_E6,579230,6142255,Curva de Maronas,UYMVD_PM2_b,Medidas de la calidad del aire - 2018,NaN,2018
36054,PM2,1,2018-01-06 14:00:00,NaN,ug/m3,UYMVD_E6,579230,6142255,Curva de Maronas,UYMVD_PM2_b,Medidas de la calidad del aire - 2018,NaN,2018
99001,PM2,1,2020-07-21 09:00:00,NaN,ug/m3,UYMVD_E5,576324,6138361,Tres Cruces 4,UYMVD_PM2_b,Medidas de la calidad del aire - 2020,66346.0,2020


#### **Tratamiento de valores nulos en `pollutant_value`**

Tras excluir la estación Tres Cruces 3 (ver sección anterior), el dataset `pm25` presenta nulos
en `pollutant_value` en un rango de 3.3% a 14.8% según la estación, con un patrón de bloques
cortos de horas consecutivas dentro de un mismo día (consistente con interrupciones breves de
sensor, no con fallas prolongadas).

Se optó por **eliminar las filas con `pollutant_value` nulo** (`dropna`), en lugar de imputarlas.
A diferencia del caso de O3 (donde el alto porcentaje de nulos hacía indeseable una imputación
por el riesgo de distorsionar la varianza), acá el argumento es más simple: el porcentaje de
nulos es bajo, y el dataset es lo suficientemente grande como para que eliminarlos no comprometa
el volumen de datos disponible para el análisis — no hay necesidad real de inventar valores
donde sobran datos reales.

In [20]:
# Se limpian los nulos de las tablas

filas_antes = len(pm25)
pm25 = pm25.dropna(subset=["pollutant_value"])
filas_despues = len(pm25)

print(f"Filas antes: {filas_antes:,}")
print(f"Filas después: {filas_despues:,}")
print(f"Filas descartadas: {filas_antes - filas_despues:,}")
print(f"Porcentaje restante: {(filas_despues / filas_antes) * 100:.2f}%")

Filas antes: 233,250
Filas después: 208,917
Filas descartadas: 24,333
Porcentaje restante: 89.57%


In [22]:
print(pm25.groupby("ID_estacion")['pollutant_value'].apply(lambda x: x.isna().mean() * 100))

ID_estacion
Ciudad Vieja 2      0.0
Ciudad Vieja 3      0.0
Colon               0.0
Curva de Maronas    0.0
Tres Cruces 4       0.0
Name: pollutant_value, dtype: float64


#### Añadir al DataFrame las columnas: `día de semana, tipo de día, hora`

In [25]:
pm25["hora_de_la_muestra"] = pm25["date"].dt.hour
pm25["dia_semana"] = pm25["date"].dt.day_name() 
pm25["tipo_dia"] = pm25["date"].dt.dayofweek.apply(lambda x: "Fin de semana" if x >= 5 else "Día de semana")
pm25 = pm25.reset_index(drop=True)

pm25.head()

,pollutant_id,pollutant_averaging,date,pollutant_value,pollutant_unit,station_id,X,Y,ID_estacion,method_id,archivo_origen,Unnamed: 0,Year,hora_de_la_muestra,dia_semana,tipo_dia
0,PM2,1,2015-01-04 00:00:00,8.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN,2015,0,Sunday,Fin de semana
1,PM2,1,2015-01-04 01:00:00,7.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN,2015,1,Sunday,Fin de semana
2,PM2,1,2015-01-04 02:00:00,6.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN,2015,2,Sunday,Fin de semana
3,PM2,1,2015-01-04 03:00:00,4.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN,2015,3,Sunday,Fin de semana
4,PM2,1,2015-01-04 04:00:00,4.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN,2015,4,Sunday,Fin de semana
